In [0]:

from pyspark.sql import functions as F


SOURCE_PATH = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/marketing_bookings/"
)

TARGET_TABLE = "voyage.bronze.marketing_bookings"


# -------------------------------------------------------------------
# Read Airbyte JSONL/GZIP raw data
# -------------------------------------------------------------------

raw_df = (
    spark.read
    .json(SOURCE_PATH)
)

print("Raw row count:", raw_df.count())

raw_df.printSchema()


# -------------------------------------------------------------------
# Flatten _airbyte_data while keeping Airbyte metadata
# -------------------------------------------------------------------

metadata_columns = [
    column
    for column in raw_df.columns
    if column != "_airbyte_data"
]

bronze_df = raw_df.select(
    F.col("_airbyte_data.*"),
    *[
        F.col(column).alias(column)
        for column in metadata_columns
    ],
)


# -------------------------------------------------------------------
# Basic inspection
# -------------------------------------------------------------------

bronze_df.printSchema()

display(
    bronze_df.limit(10)
)


# -------------------------------------------------------------------
# Write Bronze Delta table
# -------------------------------------------------------------------

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Bronze table: {TARGET_TABLE}")


# -------------------------------------------------------------------
# Validation
# -------------------------------------------------------------------

print("Bronze row count:")

display(
    spark.sql(
        f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
        """
    )
)

print("Distinct booking IDs:")

display(
    spark.sql(
        f"""
        SELECT
            COUNT(DISTINCT booking_id) AS distinct_booking_ids,
            COUNT(DISTINCT conversion_id) AS distinct_conversion_ids
        FROM {TARGET_TABLE}
        """
    )
)


print("Duplicate booking IDs in Bronze:")

display(
    spark.sql(
        f"""
        SELECT
            booking_id,
            COUNT(*) AS record_count
        FROM {TARGET_TABLE}
        GROUP BY booking_id
        HAVING COUNT(*) > 1
        ORDER BY record_count DESC
        LIMIT 20
        """
    )
)

In [0]:
display(
    spark.sql(
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT booking_id) AS distinct_booking_ids,
            COUNT(DISTINCT conversion_id) AS distinct_conversion_ids
        FROM voyage.bronze.marketing_bookings
        """
    )
)

In [0]:
display(
    spark.sql(
        """
        SELECT COUNT(*) AS orphan_conversions
        FROM voyage.bronze.marketing_bookings b
        LEFT JOIN voyage.bronze.website_events e
          ON b.conversion_id = e.event_id
        WHERE e.event_id IS NULL
        """
    )
)

display(
    spark.sql(
        """
        SELECT COUNT(*) AS orphan_sessions
        FROM voyage.bronze.marketing_bookings b
        LEFT JOIN voyage.bronze.website_sessions s
          ON b.session_id = s.session_id
        WHERE s.session_id IS NULL
        """
    )
)